In [ ]:
from pathlib import Path

def carpeta_proyecto(nombre, marca):
    """Localiza los CSV en Jupyter, Visual Studio Code y Colab."""
    from pathlib import Path
    import json

    saltar = {
        ".config", "sample_data", "drive", ".git", "__pycache__",
        "node_modules", ".ipynb_checkpoints", "site-packages",
    }

    def directorio_notebook():
        try:
            return Path(__vsc_ipynb_file__).resolve().parent
        except NameError:
            pass
        try:
            from urllib.request import urlopen
            try:
                from ipykernel.connect import get_connection_file
            except ImportError:
                from ipykernel import get_connection_file
            conexion = Path(get_connection_file())
            kernel_id = conexion.name.replace("kernel-", "").replace(".json", "")
            servidores = list(conexion.parent.glob("jpserver-*.json"))[:3]
            for servidor in servidores:
                try:
                    info = json.loads(servidor.read_text(encoding="utf-8"))
                except (OSError, json.JSONDecodeError):
                    continue
                url = str(info.get("url", "")).rstrip("/")
                raiz = info.get("root_dir")
                if not url or not raiz:
                    continue
                api = url + "/api/sessions"
                token = info.get("token") or ""
                if token:
                    api += "?token=" + token
                try:
                    with urlopen(api, timeout=0.8) as respuesta:
                        sesiones = json.loads(respuesta.read().decode())
                except Exception:
                    continue
                for sesion in sesiones:
                    kid = (sesion.get("kernel") or {}).get("id")
                    if kid != kernel_id:
                        continue
                    cuaderno = sesion.get("notebook") or {}
                    relativo = cuaderno.get("path") or sesion.get("path")
                    if relativo:
                        return (Path(raiz) / relativo).resolve().parent
        except Exception:
            pass
        return None

    bases = []
    notebook = directorio_notebook()
    if notebook is not None:
        bases.append(notebook)
    bases.append(Path.cwd().resolve())
    content = Path("/content")
    if content.is_dir():
        bases.append(content)
        drive = content / "drive" / "MyDrive"
        if drive.is_dir():
            bases.append(drive)

    nombradas = []
    sueltas = []
    vistos = set()

    def considerar(directorio, aceptar_suelto):
        if directorio in vistos:
            return
        vistos.add(directorio)
        try:
            carpeta = directorio / nombre
            if (carpeta / marca).is_file():
                nombradas.append(carpeta)
                return
            if directorio.name == nombre and (directorio / marca).is_file():
                nombradas.append(directorio)
                return
            if aceptar_suelto and (directorio / marca).is_file():
                sueltas.append(directorio)
        except OSError:
            return

    def hijos(directorio, profundidad, limite):
        if profundidad < 0 or limite[0] <= 0 or not directorio.is_dir():
            return
        try:
            entradas = list(directorio.iterdir())
        except OSError:
            return
        for hijo in entradas:
            if not hijo.is_dir() or hijo.name.startswith(".") or hijo.name in saltar:
                continue
            limite[0] -= 1
            considerar(hijo, False)
            if nombradas or limite[0] <= 0:
                return
            hijos(hijo, profundidad - 1, limite)
            if nombradas or limite[0] <= 0:
                return

    for base in bases:
        considerar(base, True)
        if nombradas:
            break
        for padre in list(base.parents)[:6]:
            considerar(padre, False)
            if nombradas:
                break
        if nombradas:
            break
        profundidad = 3 if base.name in {"content", "MyDrive"} else 2
        hijos(base, profundidad, [400])
        if nombradas:
            break

    if nombradas:
        return nombradas[0]
    if sueltas:
        return sueltas[0]
    raise FileNotFoundError(
        "No se encuentra " + marca + ". "
        "Tiene que existir la carpeta '" + nombre + "' en el proyecto, "
        "o el archivo en el directorio de trabajo. En Colab puede estar junto a sample_data."
    )




PREPROCESADOS = carpeta_proyecto("datos preprocesados", "df_macro.csv")
print("Preprocesados:", PREPROCESADOS)

import pandas as pd
import numpy as np

df_modelo_30 = pd.read_csv(PREPROCESADOS / "df_modelo_30.csv")
df_macro = pd.read_csv(PREPROCESADOS / "df_macro.csv")

# **ML**

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
# # Separar canciones en train y test

from sklearn.model_selection import train_test_split

SEED = 42
TEST_SIZE = 0.2

ids_train, ids_test = train_test_split(
    df_modelo_30['id_cancion'],
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=df_modelo_30['id_genero']
)

df_train = df_macro[df_macro['id_cancion'].isin(ids_train)].copy()
df_test = df_macro[df_macro['id_cancion'].isin(ids_test)].copy()

ids_train_set = set(df_train['id_cancion'].unique())
ids_test_set = set(df_test['id_cancion'].unique())

ids_repetidos = ids_train_set.intersection(ids_test_set)

print("IDs en train:", len(ids_train))
print("IDs en test:", len(ids_test))

print("Filas en df_train:", df_train.shape[0])
print("Filas en df_test:", df_test.shape[0])

print("Canciones únicas en df_train:", df_train['id_cancion'].nunique())
print("Canciones únicas en df_test:", df_test['id_cancion'].nunique())

print("IDs repetidos entre train y test:", len(ids_repetidos))

IDs en train: 800
IDs en test: 200
Filas en df_train: 7990
Filas en df_test: 2000
Canciones únicas en df_train: 800
Canciones únicas en df_test: 200
IDs repetidos entre train y test: 0


In [ ]:
# Mostrar IDs de train que han salido despues de la funcion.

print(ids_train)

575    576
204    205
964    965
503    504
502    503
      ... 
915    916
346    347
930    931
306    307
270    271
Name: id_cancion, Length: 800, dtype: int64


In [ ]:
# Guardamos los datos de train y test, para Ml.
ids_train.to_csv(PREPROCESADOS / "ids_train.csv", index=False)
ids_test.to_csv(PREPROCESADOS / "ids_test.csv", index=False)


In [ ]:
# Cargar IDs ya generados previamente desde CSV

ids_train = pd.read_csv(PREPROCESADOS / "ids_train.csv")['id_cancion']
ids_test = pd.read_csv(PREPROCESADOS / "ids_test.csv")['id_cancion']

print("IDs train:", len(ids_train))
print("IDs test:", len(ids_test))

IDs train: 800
IDs test: 200


In [ ]:
# Filtrar tabla final por IDs de train y test y los mostramos, no hay ningún id en train que se repita en test.

ids_train = df_macro[df_macro['id_cancion'].isin(ids_train)].copy()
ids_test = df_macro[df_macro['id_cancion'].isin(ids_test)].copy()

display(df_train.head(30))
display(df_test.head(30))

,id_cancion,filename_3s,length_3s,chroma_stft_mean_3s,chroma_stft_var_3s,rms_mean_3s,rms_var_3s,spectral_centroid_mean_3s,spectral_centroid_var_3s,spectral_bandwidth_mean_3s,...,mfcc16_var_30s,mfcc17_mean_30s,mfcc17_var_30s,mfcc18_mean_30s,mfcc18_var_30s,mfcc19_mean_30s,mfcc19_var_30s,mfcc20_mean_30s,mfcc20_var_30s,label_30s
0,1,blues.00000.0.wav,66149,0.335406,0.091048,0.130405,0.003521,1773.065032,167541.630869,1972.744388,...,52.420910,-1.690215,36.524071,-0.408979,41.597103,-2.303523,55.062923,1.221291,46.936035,blues
1,1,blues.00000.1.wav,66149,0.343065,0.086147,0.112699,0.001450,1816.693777,90525.690866,2010.051501,...,52.420910,-1.690215,36.524071,-0.408979,41.597103,-2.303523,55.062923,1.221291,46.936035,blues
2,1,blues.00000.2.wav,66149,0.346815,0.092243,0.132003,0.004620,1788.539719,111407.437613,2084.565132,...,52.420910,-1.690215,36.524071,-0.408979,41.597103,-2.303523,55.062923,1.221291,46.936035,blues
3,1,blues.00000.3.wav,66149,0.363639,0.086856,0.132565,0.002448,1655.289045,111952.284517,1960.039988,...,52.420910,-1.690215,36.524071,-0.408979,41.597103,-2.303523,55.062923,1.221291,46.936035,blues
4,1,blues.00000.4.wav,66149,0.335579,0.088129,0.143289,0.001701,1630.656199,79667.267654,1948.503884,...,52.420910,-1.690215,36.524071,-0.408979,41.597103,-2.303523,55.062923,1.221291,46.936035,blues
5,1,blues.00000.5.wav,66149,0.376670,0.089702,0.132618,0.003583,1994.915219,211700.619569,2152.767854,...,52.420910,-1.690215,36.524071,-0.408979,41.597103,-2.303523,55.062923,1.221291,46.936035,blues
6,1,blues.00000.6.wav,66149,0.379909,0.088827,0.130335,0.003166,1962.150096,177443.070045,2146.503479,...,52.420910,-1.690215,36.524071,-0.408979,41.597103,-2.303523,55.062923,1.221291,46.936035,blues
7,1,blues.00000.7.wav,66149,0.331880,0.092119,0.140600,0.002546,1701.890924,35678.130616,1979.387612,...,52.420910,-1.690215,36.524071,-0.408979,41.597103,-2.303523,55.062923,1.221291,46.936035,blues
8,1,blues.00000.8.wav,66149,0.347877,0.094209,0.133130,0.002538,1746.473502,138073.931244,1887.619723,...,52.420910,-1.690215,36.524071,-0.408979,41.597103,-2.303523,55.062923,1.221291,46.936035,blues
9,1,blues.00000.9.wav,66149,0.358061,0.082957,0.115312,0.001846,1763.948942,61493.423121,1874.195710,...,52.420910,-1.690215,36.524071,-0.408979,41.597103,-2.303523,55.062923,1.221291,46.936035,blues


,id_cancion,filename_3s,length_3s,chroma_stft_mean_3s,chroma_stft_var_3s,rms_mean_3s,rms_var_3s,spectral_centroid_mean_3s,spectral_centroid_var_3s,spectral_bandwidth_mean_3s,...,mfcc16_var_30s,mfcc17_mean_30s,mfcc17_var_30s,mfcc18_mean_30s,mfcc18_var_30s,mfcc19_mean_30s,mfcc19_var_30s,mfcc20_mean_30s,mfcc20_var_30s,label_30s
10,2,blues.00001.0.wav,66149,0.402401,0.090340,0.093024,0.003876,1279.182242,406513.816744,1921.306192,...,55.356403,-0.731125,60.314529,0.295073,48.120598,-0.283518,51.10619,0.531217,45.786282,blues
11,2,blues.00001.1.wav,66149,0.345507,0.091037,0.094656,0.001495,1513.763969,214768.804443,2091.432928,...,55.356403,-0.731125,60.314529,0.295073,48.120598,-0.283518,51.10619,0.531217,45.786282,blues
12,2,blues.00001.2.wav,66149,0.338119,0.083682,0.097776,0.001386,1308.869728,154209.122448,1508.008806,...,55.356403,-0.731125,60.314529,0.295073,48.120598,-0.283518,51.10619,0.531217,45.786282,blues
13,2,blues.00001.3.wav,66149,0.330751,0.093936,0.085365,0.002641,1479.530348,646868.373151,1906.454247,...,55.356403,-0.731125,60.314529,0.295073,48.120598,-0.283518,51.10619,0.531217,45.786282,blues
14,2,blues.00001.4.wav,66149,0.348027,0.100096,0.088437,0.002204,1729.640428,568827.519152,2150.738315,...,55.356403,-0.731125,60.314529,0.295073,48.120598,-0.283518,51.10619,0.531217,45.786282,blues
15,2,blues.00001.5.wav,66149,0.329435,0.104149,0.093201,0.001929,1492.089309,228951.553214,2189.638968,...,55.356403,-0.731125,60.314529,0.295073,48.120598,-0.283518,51.10619,0.531217,45.786282,blues
16,2,blues.00001.6.wav,66149,0.343666,0.096098,0.089783,0.002173,1571.420445,257630.094708,2183.513315,...,55.356403,-0.731125,60.314529,0.295073,48.120598,-0.283518,51.10619,0.531217,45.786282,blues
17,2,blues.00001.7.wav,66149,0.327732,0.099648,0.110633,0.003306,1877.638297,510311.370648,2326.810264,...,55.356403,-0.731125,60.314529,0.295073,48.120598,-0.283518,51.10619,0.531217,45.786282,blues
18,2,blues.00001.8.wav,66149,0.329428,0.090254,0.111013,0.001072,1665.600276,239973.571541,2168.031814,...,55.356403,-0.731125,60.314529,0.295073,48.120598,-0.283518,51.10619,0.531217,45.786282,blues
19,2,blues.00001.9.wav,66149,0.292987,0.082145,0.094924,0.003031,1373.437553,210606.446462,1929.319087,...,55.356403,-0.731125,60.314529,0.295073,48.120598,-0.283518,51.10619,0.531217,45.786282,blues


In [ ]:
# Revisar IDs repetidos entre train y test

ids_train_set = set(df_train['id_cancion'].unique())
ids_test_set = set(df_test['id_cancion'].unique())

ids_repetidos = ids_train_set.intersection(ids_test_set)

print("IDs repetidos entre train y test:", len(ids_repetidos))

IDs repetidos entre train y test: 0


In [ ]:
# Filtramos las canciones del DataFrame original para obtener solo las canciones que están en los conjuntos de entrenamiento y prueba.
df_train_genero = df_modelo_30[df_modelo_30['id_cancion'].isin(ids_train)]
df_test_genero = df_modelo_30[df_modelo_30['id_cancion'].isin(ids_test)]

print("df_train_genero:", df_train_genero.shape)
print("df_test_genero:", df_test_genero.shape)

display(df_train_genero.head())
display(df_test_genero.head())

df_train_genero: (800, 68)
df_test_genero: (200, 68)


,id_cancion,cancion,duracion,coste,id_genero,id_disco,id_autor,id_discografica,filename_30s,length_30s,...,mfcc16_var_30s,mfcc17_mean_30s,mfcc17_var_30s,mfcc18_mean_30s,mfcc18_var_30s,mfcc19_mean_30s,mfcc19_var_30s,mfcc20_mean_30s,mfcc20_var_30s,label_30s
0,1,blues.00000.wav,138.0,744.01,1,1,1,1,blues.00000.wav,661794,...,52.420910,-1.690215,36.524071,-0.408979,41.597103,-2.303523,55.062923,1.221291,46.936035,blues
3,4,blues.00003.wav,111.0,3618.77,1,4,6,4,blues.00003.wav,661794,...,44.427753,-3.319597,50.206673,0.636965,37.319130,-0.619121,37.259739,-3.407448,31.949339,blues
4,5,blues.00004.wav,191.0,1990.81,1,5,7,5,blues.00004.wav,661794,...,86.099236,-5.454034,75.269707,-0.916874,53.613918,-4.404827,62.910812,-11.703234,55.195160,blues
5,6,blues.00005.wav,231.0,909.81,1,6,9,6,blues.00005.wav,661794,...,72.549225,-1.838263,68.702026,-2.783800,42.447453,-3.047909,39.808784,-8.109991,46.311005,blues
6,7,blues.00006.wav,160.0,2311.44,1,6,11,7,blues.00006.wav,661794,...,83.248245,-10.913176,56.902153,-6.971336,38.231800,-3.436505,48.235741,-6.483466,70.170364,blues


,id_cancion,cancion,duracion,coste,id_genero,id_disco,id_autor,id_discografica,filename_30s,length_30s,...,mfcc16_var_30s,mfcc17_mean_30s,mfcc17_var_30s,mfcc18_mean_30s,mfcc18_var_30s,mfcc19_mean_30s,mfcc19_var_30s,mfcc20_mean_30s,mfcc20_var_30s,label_30s
1,2,blues.00001.wav,142.0,4178.85,1,2,2,2,blues.00001.wav,661794,...,55.356403,-0.731125,60.314529,0.295073,48.120598,-0.283518,51.106190,0.531217,45.786282,blues
2,3,blues.00002.wav,96.0,3226.40,1,3,4,3,blues.00002.wav,661794,...,40.598766,-7.729093,47.639427,-1.816407,52.382141,-3.439720,46.639660,-2.231258,30.573025,blues
14,15,blues.00014.wav,69.0,4224.17,1,10,20,1,blues.00014.wav,661794,...,67.754112,0.054863,96.668579,-3.722265,65.745132,-0.994969,89.352570,-1.800289,113.520569,blues
20,21,blues.00020.wav,178.0,3826.86,1,13,21,16,blues.00020.wav,661794,...,50.114544,-4.500184,52.987198,-1.822338,73.484642,-0.517100,94.666908,1.509918,73.787224,blues
21,22,blues.00021.wav,169.0,2076.21,1,12,14,7,blues.00021.wav,661794,...,89.542923,-3.680215,93.907761,0.491501,119.286835,5.041454,117.804779,-0.776849,148.473969,blues


In [ ]:
# Calculamos el número de canciones por género en total, train y test

train_genero_counts = df_train_genero['id_genero'].value_counts().sort_index()
test_genero_counts = df_test_genero['id_genero'].value_counts().sort_index()
total_genero_counts = df_modelo_30['id_genero'].value_counts().sort_index()

# Calculamos porcentajes respecto al total de cada género

percentage_train_genero = (train_genero_counts / total_genero_counts) * 100
percentage_test_genero = (test_genero_counts / total_genero_counts) * 100

# Mostramos el reparto final por género

df_percentages = pd.DataFrame({
    'Total_Canciones': total_genero_counts,
    'Train_Canciones': train_genero_counts,
    'Test_Canciones': test_genero_counts,
    'Porcentaje_Train': percentage_train_genero,
    'Porcentaje_Test': percentage_test_genero
}).fillna(0)

display(df_percentages.round(2))

,Total_Canciones,Train_Canciones,Test_Canciones,Porcentaje_Train,Porcentaje_Test
id_genero,,,,,
1,100,80,20,80.0,20.0
2,100,80,20,80.0,20.0
3,100,80,20,80.0,20.0
4,100,80,20,80.0,20.0
5,100,80,20,80.0,20.0
6,100,80,20,80.0,20.0
7,100,80,20,80.0,20.0
8,100,80,20,80.0,20.0
9,100,80,20,80.0,20.0
